<a href="https://colab.research.google.com/github/Lalitha1704/Football_Video_Summarization_Using_DL/blob/main/Football_Video_Summarization_Using_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics opencv-python moviepy

In [ ]:
from google.colab import files
uploaded = files.upload()

VIDEO_PATH = list(uploaded.keys())[0]
print("Input video:", VIDEO_PATH)

In [ ]:
import cv2
from ultralytics import YOLO
from moviepy.editor import VideoFileClip, concatenate_videoclips

model = YOLO("yolov8x.pt")

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

important_times = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=0.4, verbose=False)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])

            # COCO class 32 = sports ball
            if cls == 32:
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # Assume goal area is right side
                if x2 > frame_width * 0.7:
                    time_sec = frame_count // fps
                    important_times.append(time_sec)

    frame_count += 1

cap.release()

important_times = sorted(set(important_times))
print("Raw detected times:", important_times)

In [ ]:
if len(important_times) == 0:
    print("No goal attempts detected")
    exit()

GROUP_GAP = 5  # seconds

grouped_times = []
current_group = [important_times[0]]

for t in important_times[1:]:
    if t - current_group[-1] <= GROUP_GAP:
        current_group.append(t)
    else:
        grouped_times.append(current_group)
        current_group = [t]

grouped_times.append(current_group)

event_times = [group[len(group)//2] for group in grouped_times]
print("Final goal attempts:", event_times)

In [ ]:
POST_EVENT_CHECK = 3      # seconds
GOAL_REGION_RATIO = 0.85 # deeper goal area
MIN_FRAMES_IN_GOAL = 5   # persistence threshold


def is_goal(video_path, event_time, fps, frame_width):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(event_time * fps))

    frames_to_check = int(4 * fps)

    positions = []
    detected_frames = 0

    for _ in range(frames_to_check):
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.4, verbose=False)

        for r in results:
            for box in r.boxes:
                if int(box.cls[0]) == 32:  # sports ball
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cx = (x1 + x2) // 2
                    positions.append(cx)
                    detected_frames += 1

    cap.release()

    # -------- HEURISTICS --------

    if detected_frames == 0:
        return False  # No info → MISS

    # Deep goal region presence
    deep_goal = any(x > frame_width * 0.85 for x in positions)

    # Sudden slowdown (ball stops)
    if len(positions) >= 3:
        speeds = [abs(positions[i+1] - positions[i]) for i in range(len(positions)-1)]
        slowdown = min(speeds) < 3
    else:
        slowdown = False

    # Ball disappears early
    disappearance = detected_frames < frames_to_check * 0.3

    # Decision rule (any 2 signals)
    score = sum([deep_goal, slowdown, disappearance])

    return score >= 2


In [ ]:
event_results = []  # (time, label)

for t in event_times:
    label = "GOAL" if is_goal(VIDEO_PATH, t, fps, frame_width) else "MISS"
    event_results.append((t, label))

print("\nEvent classification:")
for t, label in event_results:
    print(f"Time {t}s → {label}")

In [ ]:
BUFFER_BEFORE = 3
BUFFER_AFTER = 3
goal_segments = []
miss_segments = []

for t, label in event_results:
    start = max(0, t - BUFFER_BEFORE)
    end = t + BUFFER_AFTER

    if label == "GOAL":
        goal_segments.append((start, end))
    else:
        miss_segments.append((start, end))


In [ ]:
def merge_segments(segments):
    if not segments:
        return []

    segments = sorted(segments)
    merged = [list(segments[0])]

    for start, end in segments[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1][1] = max(last_end, end)
        else:
            merged.append([start, end])

    return merged
goal_segments = merge_segments(goal_segments)
miss_segments = merge_segments(miss_segments)

print("GOAL segments:", goal_segments)
print("MISS segments:", miss_segments)



# -----------------------------
# STEP 1: Convert segments → predicted events
# -----------------------------
pred_event_results = []

for start, end in goal_segments:
    pred_event_results.append(((start, end), "Goal"))

for start, end in miss_segments:
    pred_event_results.append(((start, end), "Miss"))

# Sort by time
pred_event_results = sorted(pred_event_results, key=lambda x: x[0][0])

print("\nPredicted Events:")
for ev in pred_event_results:
    print(ev)

# print("\nPredicted Events:")
# for ev in pred_event_results:
#     print(ev)




# -----------------------------
# 👉 ADD CSV CREATION HERE
# -----------------------------
import pandas as pd

data = []

# Goal → 1
for start, end in goal_segments:
    data.append([start, end, 1])

# Miss → 0
for start, end in miss_segments:
    data.append([start, end, 0])

df = pd.DataFrame(data, columns=["start_time", "end_time", "label"])
df = df.sort_values(by="start_time")

df.to_csv("predicted_events.csv", index=False)

print("\nCSV created successfully!")

from google.colab import files
files.download("predicted_events.csv")

def match_segments(pred_events, true_events):
    y_true = []
    y_pred = []

    for (p_start, p_end), p_label in pred_events:
        matched = False

        for (t_start, t_end), t_label in true_events:
            if not (p_end < t_start or p_start > t_end):  # overlap
                y_true.append(label_map[t_label.lower()])
                y_pred.append(label_map[p_label.lower()])
                matched = True
                break

        if not matched:
            y_true.append(0)
            y_pred.append(label_map[p_label.lower()])

    return y_true, y_pred


In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
from google.colab import files

# -----------------------------
# MERGE FUNCTION (same as yours)
# -----------------------------
def merge_segments(segments):
    if not segments:
        return []

    segments = sorted(segments)
    merged = [list(segments[0])]

    for start, end in segments[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1][1] = max(last_end, end)
        else:
            merged.append([start, end])

    return merged


# -----------------------------
# PREPARE SEGMENTS
# -----------------------------
BUFFER_BEFORE = 3
BUFFER_AFTER = 3

goal_segments = []
miss_segments = []

for t, label in event_results:
    start = max(0, t - BUFFER_BEFORE)
    end = t + BUFFER_AFTER

    if label == "GOAL":
        goal_segments.append((start, end))
    else:
        miss_segments.append((start, end))

goal_segments = merge_segments(goal_segments)
miss_segments = merge_segments(miss_segments)

# -----------------------------
# COMBINE ALL SEGMENTS
# -----------------------------
all_segments = goal_segments + miss_segments
all_segments = merge_segments(all_segments)   # Merge again if overlaps exist
all_segments = sorted(all_segments)

print("Final segments:", all_segments)

# -----------------------------
# CREATE SINGLE VIDEO
# -----------------------------
video = VideoFileClip(VIDEO_PATH)

final_clips = [video.subclip(start, end) for start, end in all_segments]

if final_clips:
    final_video = concatenate_videoclips(final_clips)
    final_video.write_videofile(
        "event_summary.mp4",
        codec="libx264",
        audio_codec="aac"
    )

    files.download("event_summary.mp4")

In [ ]:
# -----------------------------
# LOAD GROUND TRUTH FROM CSV
# -----------------------------
from google.colab import files
import pandas as pd

# -----------------------------
# STEP 1: Upload file manually
# -----------------------------
uploaded = files.upload()

# Get uploaded file name
file_name = list(uploaded.keys())[0]

# -----------------------------
# STEP 2: Read CSV
# -----------------------------
df = pd.read_csv(file_name)

# -----------------------------
# STEP 3: Convert to required format
# -----------------------------
true_event_results = []

for _, row in df.iterrows():
    label = "Goal" if row["label"] == 1 else "Miss"
    true_event_results.append(
        ((int(row["start_time"]), int(row["end_time"])), label)
    )

print("\nLoaded Ground Truth from CSV:")
for ev in true_event_results:
    print(ev)


# -----------------------------
# METRICS CALCULATION
# -----------------------------
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

label_map = {"miss": 0, "goal": 1}

# Compare predictions vs ground truth
y_true, y_pred = match_segments(pred_event_results, true_event_results)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Handle edge case (if only one class present)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
else:
    tn = fp = fn = tp = 0

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# -----------------------------
# PRINT RESULTS
# -----------------------------
print("\nConfusion Matrix:")
print(cm)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))


# -----------------------------
# PLOT HEATMAP
# -----------------------------
plt.figure(figsize=(6,5))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["MISS (0)", "GOAL (1)"],
            yticklabels=["MISS (0)", "GOAL (1)"])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

if cm.shape == (2, 2):
    plt.title(f"Confusion Matrix\n(TP={tp}, FP={fp}, FN={fn}, TN={tn})")
else:
    plt.title("Confusion Matrix")

plt.show()